# English → Arabic Translation Evaluation (Helsinki-NLP, fine-tuned)

This notebook:
1. Loads your fine-tuned Helsinki-NLP MarianMT model from `/kaggle/input/`
2. Loads the English–Arabic `devtest.csv` parallel corpus
3. Translates every English sentence to Arabic
4. Scores the translations with **BLEU** and **COMET**

Each step is in its own cell/function, so you can re-run just the piece you need (e.g. re-score without re-translating).


## 1. Install dependencies

⚠️ Make sure **Internet** is turned on in the notebook's right-hand settings panel (needed for pip installs and downloading the COMET checkpoint).

In [1]:
!pip install -q sacrebleu unbabel-comet sentencepiece sacremoses --no-warn-conflicts


## 2. Imports

In [2]:
import os
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer
from tqdm.auto import tqdm
import sacrebleu

print("Torch sees GPU:", torch.cuda.is_available())


2026-07-12 08:57:29.080513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783846649.497285     164 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783846649.624597     164 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783846650.664421     164 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783846650.664463     164 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783846650.664466     164 computation_placer.cc:177] computation placer alr

Torch sees GPU: True


## 3. Config — 🔧 customize these paths

Kaggle input paths can vary depending on how the dataset/model was attached. If any path below throws a "not found" error, run the **directory listing cell (3b)** below first to see the exact folder names, then fix the paths here.


In [3]:
# 🔧 CUSTOMIZE: path to the devtest CSV (English–Arabic parallel corpus)
DEVTEST_CSV_PATH = "/kaggle/input/datasets/mishbhaul/english-arabic-devtest/devtest.csv"

# 🔧 CUSTOMIZE: column names in devtest.csv holding English and Arabic sentences
SRC_COL = "en"   # source language column (English)
TGT_COL = "ar"    # reference/target column (Arabic)

# 🔧 CUSTOMIZE: path to the fine-tuned Helsinki model folder
# (this should be a folder containing config.json, tokenizer files, and model weights)
MODEL_DIR = "/kaggle/input/models/mishbhaul/hensinki-en-ar-final/transformers/default/1/helsinki-en-ar-final"

# 🔧 CUSTOMIZE: translation / evaluation settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
MAX_LENGTH = 128

# 🔧 CUSTOMIZE: which COMET checkpoint to use
# "Unbabel/wmt22-comet-da" is the standard, widely-reported COMET model
COMET_MODEL_NAME = "Unbabel/wmt22-comet-da"


### 3b. (Optional) List `/kaggle/input` contents — run this if a path above is wrong

In [4]:
for root, dirs, files in os.walk("/kaggle/input"):
    # only go 3 levels deep so this doesn't flood the output
    depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
    if depth > 3:
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


input/
  datasets/
    mishbhaul/
      english-arabic-devtest/
        devtest.csv
  models/
    mishbhaul/
      hensinki-en-ar-final/


## 4. Load the devtest dataset

In [5]:
def load_devtest(csv_path, src_col, tgt_col):
    """Loads the parallel corpus and does a quick sanity check on the columns."""
    df = pd.read_csv(csv_path)

    if src_col not in df.columns or tgt_col not in df.columns:
        raise ValueError(
            f"Expected columns '{src_col}' and '{tgt_col}' not found. "
            f"Available columns are: {list(df.columns)} — update SRC_COL/TGT_COL above."
        )

    # drop any rows with missing text, just in case
    df = df.dropna(subset=[src_col, tgt_col]).reset_index(drop=True)
    return df


devtest_df = load_devtest(DEVTEST_CSV_PATH, SRC_COL, TGT_COL)
print(f"Loaded {len(devtest_df)} sentence pairs")
devtest_df.head()


Loaded 500 sentence pairs


,split,en,ar
0,devtest,Policy makers will gather tomorrow in Al-Ula t...,ويجتمع يوم غدٍ صناع السياسات في العلا لتبني ال...
1,devtest,After defeating Marseille 2-1 in their opening...,ورفع الفريق الإسباني بطل أوروبا 15 مرة رصيده إ...
2,devtest,"The CEO and founder, Carl Pei, personally shar...",الرئيس التنفيذي والمؤسس كارل بي شارك بنفسه بتط...
3,devtest,"According to the Saudi Ministry of Finance, th...",توقعت وزارة المالية السعودية أن يسجل الاقتصاد ...
4,devtest,According to estimates from the Saudi General ...,كما أظهرت تقديرات الهيئة العامة للإحصاء السعود...


## 5. Load the fine-tuned model and tokenizer

In [6]:
def load_model(model_dir, device):
    """Loads a MarianMT model + tokenizer from a local folder."""
    tokenizer = MarianTokenizer.from_pretrained(model_dir)
    model = MarianMTModel.from_pretrained(model_dir)
    model.to(device)
    model.eval()
    return model, tokenizer


model, tokenizer = load_model(MODEL_DIR, DEVICE)
print(f"Model loaded on {DEVICE}")


Model loaded on cuda


## 6. Translation function

In [7]:
@torch.no_grad()
def translate_batch(sentences, model, tokenizer, device, max_length=MAX_LENGTH):
    """Translates a list of source sentences in one forward pass."""
    inputs = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    ).to(device)

    generated = model.generate(**inputs, max_length=max_length)
    translations = tokenizer.batch_decode(generated, skip_special_tokens=True)
    return translations


def translate_all(sentences, model, tokenizer, device, batch_size=BATCH_SIZE):
    """Runs translate_batch over the whole dataset in chunks, with a progress bar."""
    all_translations = []
    for i in tqdm(range(0, len(sentences), batch_size), desc="Translating"):
        batch = sentences[i:i + batch_size]
        translations = translate_batch(batch, model, tokenizer, device)
        all_translations.extend(translations)
    return all_translations


## 7. Run translation over the full devtest set

In [8]:
source_sentences = devtest_df[SRC_COL].tolist()
reference_sentences = devtest_df[TGT_COL].tolist()

predictions = translate_all(source_sentences, model, tokenizer, DEVICE)

# keep everything together for inspection / saving
results_df = pd.DataFrame({
    "source_en": source_sentences,
    "reference_ar": reference_sentences,
    "prediction_ar": predictions,
})
results_df.head()


Translating:   0%|          | 0/32 [00:00<?, ?it/s]

,source_en,reference_ar,prediction_ar
0,Policy makers will gather tomorrow in Al-Ula t...,ويجتمع يوم غدٍ صناع السياسات في العلا لتبني ال...,ويجتمع صانعو السياسات غدًا في العلا لبناء المد...
1,After defeating Marseille 2-1 in their opening...,ورفع الفريق الإسباني بطل أوروبا 15 مرة رصيده إ...,وبعد فوزه على مرسيليا 2-1 في مباراته الافتتاحي...
2,"The CEO and founder, Carl Pei, personally shar...",الرئيس التنفيذي والمؤسس كارل بي شارك بنفسه بتط...,شارك الرئيس التنفيذي والمؤسس، كارل بي، تطبيقًا...
3,"According to the Saudi Ministry of Finance, th...",توقعت وزارة المالية السعودية أن يسجل الاقتصاد ...,وتوقعت وزارة المالية السعودية أن يسجل الاقتصاد...
4,According to estimates from the Saudi General ...,كما أظهرت تقديرات الهيئة العامة للإحصاء السعود...,ووفقًا لتقديرات الهيئة العامة للإحصاء السعودية...


In [9]:
# save predictions so you don't have to re-translate if you just want to re-score later
results_df.to_csv("/kaggle/working/en_ar_predictions.csv", index=False)
print("Saved predictions to /kaggle/working/en_ar_predictions.csv")


Saved predictions to /kaggle/working/en_ar_predictions.csv


## 8. Compute BLEU (via sacrebleu)

In [10]:
def compute_bleu(predictions, references):
    """
    sacrebleu expects references as a list of reference-lists
    (supports multiple references per sentence — we only have one here).
    """
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    return bleu.score


bleu_score = compute_bleu(results_df["prediction_ar"].tolist(), results_df["reference_ar"].tolist())
print(f"BLEU: {bleu_score:.4f}")


BLEU: 22.0033


## 9. Compute COMET

COMET needs [source, prediction, reference] triplets — unlike BLEU, it actually looks at the *source* sentence too, not just prediction vs. reference.


In [11]:
from comet import download_model, load_from_checkpoint

def load_comet(model_name=COMET_MODEL_NAME):
    model_path = download_model(model_name)
    comet_model = load_from_checkpoint(model_path)
    return comet_model


def compute_comet(comet_model, sources, predictions, references, batch_size=BATCH_SIZE):
    data = [
        {"src": s, "mt": p, "ref": r}
        for s, p, r in zip(sources, predictions, references)
    ]
    output = comet_model.predict(data, batch_size=batch_size, gpus=1 if DEVICE == "cuda" else 0)
    return output.system_score, output.scores  # overall score, and per-sentence scores


comet_model = load_comet()
comet_score, comet_per_sentence = compute_comet(
    comet_model,
    results_df["source_en"].tolist(),
    results_df["prediction_ar"].tolist(),
    results_df["reference_ar"].tolist(),
)
print(f"COMET: {comet_score:.4f}")


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packag

COMET: 0.8777


## 10. Final summary

In [12]:
results_df["comet_score"] = comet_per_sentence

print("=" * 40)
print("EVALUATION SUMMARY")
print("=" * 40)
print(f"Sentences evaluated : {len(results_df)}")
print(f"BLEU                : {bleu_score:.4f}")
print(f"COMET               : {comet_score:.4f}")
print("=" * 40)

results_df.to_csv("/kaggle/working/en_ar_predictions_scored.csv", index=False)
print("Saved full results (with per-sentence COMET) to /kaggle/working/en_ar_predictions_scored.csv")


EVALUATION SUMMARY
Sentences evaluated : 500
BLEU                : 22.0033
COMET               : 0.8777
Saved full results (with per-sentence COMET) to /kaggle/working/en_ar_predictions_scored.csv


## Notes

- **First run tips**: run cell 3b first if `MODEL_DIR` or `DEVTEST_CSV_PATH` throw a "not found" error — it prints the real folder names so you can fix the paths above.
- **Re-scoring without re-translating**: if you already have `en_ar_predictions.csv` saved, you can skip straight to Section 8/9 by loading it with `results_df = pd.read_csv("/kaggle/working/en_ar_predictions.csv")` instead of re-running the translation loop.
- **Batch size**: if you hit a CUDA out-of-memory error, lower `BATCH_SIZE` in the config cell (try 8 or 4).
- **Adding more metrics**: chrF2 and TER are one-liners with `sacrebleu.corpus_chrf(...)` / `sacrebleu.corpus_ter(...)` if you want them alongside BLEU — happy to add these if needed.
